# BugBuster — Development Environment Setup

**Architecture:** BBP v4 wire protocol · `bugbuster` Python library (USB + HTTP transports) · PlatformIO ESP32-S3 firmware · CMake/Ninja RP2040 HAT firmware · Tauri v2 + Leptos desktop app  
**Last refreshed:** 2026-05-04

This notebook verifies and configures the full BugBuster development environment.
Run cells top-to-bottom once per machine; subsequent sessions only need the venv activated.

| Section | What it does |
|---|---|
| 1 | Prerequisites overview |
| 2 | Pre-flight tool check |
| 3 | Create venv + install Python library (editable) |
| 4 | Install test dependencies |
| 5 | Verify installation |
| 6 | Run unit test suite (no hardware) |

## 1. Prerequisites

Install the following tools via your OS package manager **before** running this notebook.
The notebook checks for them but does **not** install system-level tools automatically.

| Tool | Purpose | Install reference |
|---|---|---|
| Python 3.11+ | Library + tests | [python.org/downloads](https://www.python.org/downloads/) |
| Rust + cargo | Desktop app (Tauri v2) | [rustup.rs](https://rustup.rs/) |
| PlatformIO CLI | ESP32-S3 firmware | [docs.platformio.org](https://docs.platformio.org/en/latest/core/installation/) |
| arm-none-eabi-gcc | RP2040 firmware | [developer.arm.com](https://developer.arm.com/downloads/-/arm-gnu-toolchain-downloads) |
| cmake + ninja | RP2040 build system | `brew install cmake ninja` |
| Node.js + pnpm | ESP32 web UI | [nodejs.org](https://nodejs.org/) + `npm i -g pnpm` |

> **macOS note:** Homebrew arm-gcc 15.x does not include a sysroot — use the ARM-provided binary linked above.

## 2. Pre-flight tool check

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Repo root is one level up from Notebooks/
REPO_ROOT = Path(os.getcwd()).parent
VENV_DIR  = REPO_ROOT / '.venv'
print(f'Repo root: {REPO_ROOT}')

In [ ]:
!python --version
!python -c "import sys; ok = sys.version_info >= (3, 11); print('Python 3.11+ OK' if ok else 'WARN: Python 3.11+ required')"
!cargo --version || echo 'cargo missing — install Rust from https://rustup.rs/'
!pio --version || echo 'pio missing — install PlatformIO CLI'
!cmake --version | head -1 || echo 'cmake missing'
!ninja --version || echo 'ninja missing'
!node --version || echo 'node missing'
!pnpm --version || echo 'pnpm missing — npm i -g pnpm'
!arm-none-eabi-gcc --version | head -1 || echo 'arm-none-eabi-gcc missing'
!pytest --version || echo 'pytest missing (will install below)'

## 3. Create venv + install Python library

Creates `.venv/` in the repo root (if not already present), then installs the `bugbuster`
library in editable mode with the optional `[mcp]` extra (adds `mcp` + `pydantic` for
the MCP server).

> **Shell quoting:** zsh expands `[mcp]` as a glob — always quote: `pip install -e 'python/[mcp]'`

In [ ]:
# Create venv if needed
if not VENV_DIR.exists():
    print('Creating .venv ...')
    subprocess.run([sys.executable, '-m', 'venv', str(VENV_DIR)], check=True)
    print('Done.')
else:
    print(f'Venv already exists at {VENV_DIR}')

pip = str(VENV_DIR / 'bin' / 'pip')

# Upgrade pip
subprocess.run([pip, 'install', '--quiet', '--upgrade', 'pip'], check=True)

# Install bugbuster in editable mode with mcp extra
lib_path = str(REPO_ROOT / 'python') + '[mcp]'
subprocess.run([pip, 'install', '--quiet', '-e', lib_path], check=True)
print('bugbuster library installed (editable, with mcp extra).')

## 4. Install test dependencies

In [ ]:
req_file = REPO_ROOT / 'tests' / 'requirements-test.txt'
subprocess.run([pip, 'install', '--quiet', '-r', str(req_file)], check=True)
print('Test dependencies installed.')

## 5. Verify installation

In [ ]:
venv_py = str(VENV_DIR / 'bin' / 'python')
!{venv_py} --version
!{venv_py} -c "import bugbuster; print('bugbuster', bugbuster.__version__)"
!{venv_py} -m pytest --version

## 6. Run the unit test suite

The unit suite runs entirely without hardware. `PYTHONPATH=python` makes the editable
library importable from the repo root.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_ROOT / 'python')

result = subprocess.run(
    [str(VENV_DIR / 'bin' / 'python'), '-m', 'pytest',
     str(REPO_ROOT / 'tests' / 'unit'), '-q', '--maxfail=3'],
    cwd=str(REPO_ROOT),
    env=env,
)
print('\nReturn code:', result.returncode)

### Quick shell equivalents

```bash
# From repo root, venv activated:
PYTHONPATH=python pytest tests/unit -q --maxfail=3

# Full simulator suite (no hardware):
PYTHONPATH=python pytest tests/device --sim -q

# Hardware tests:
python tests/run_tests.py --usb /dev/cu.usbmodem1234
python tests/run_tests.py --http 192.168.4.1
```